full-data K-fold bagged LightGBM + XGBoost ensemble
UCS3502 Principles of Machine Learning — Assignment (CarbonGlobe / IEEE BigData Cup 2026)

**What changed vs the v3 notebook you had, and why each change should move the leaderboard score:**

1. **Uses ~16x more labeled training data.** The `zhihaow/carbonglobe` dataset ships its own
   internal `train.csv` / `val.csv` / `test.csv` site split (3,035 / 338 / 50,779 sites) — that
   split exists so the *paper* can report a clean benchmark number. It is **not** the Kaggle
   competition's hidden test set (that's the separate `ssp126`/`ssp585` future-climate files).
   v3 only ever trained on the 3,035 "train" sites and threw away the other ~50,000 labeled
   sites. This version trains on **all ~54k labeled sites** and holds out folds only for
   honest cross-validation, not for permanently discarding data. This is very likely the
   single biggest lever available here — more than any hyperparameter tweak.
2. **K-fold bagging instead of a single train/val split.** Instead of one LightGBM + one XGBoost
   model fit on one split, this trains `NUM_FOLDS` models per (target, model family) on grouped
   folds and **averages their test-set predictions**. Bagging across folds is one of the most
   reliable, low-risk score improvements in tabular Kaggle competitions — it reduces variance
   without adding a new failure mode.
3. **Honest out-of-fold (OOF) validation.** The overfitting check, the blend-weight search, and
   the feature-importance / error-vs-age diagnostics are now all computed on OOF predictions —
   predictions from a model that never saw that site during training — instead of a single
   held-out split. That removes the last bit of optimistic bias.
4. **XGBoost is now tuned with the same rigor as LightGBM.** v3 only grid-searched LightGBM;
   XGBoost used one fixed, comparatively weak set of regularization params. Both families now
   go through the same grouped-CV search.
5. **New features aimed at the actual difficulty of this task: extrapolating to future,
   higher-CO2 climate scenarios.** Tree models cannot extrapolate past the numeric ranges they
   were trained on. Training forcing is present-day; `ssp585` in particular pushes CO2 and
   temperature drivers well outside that range by year 40. Two additions help some (not a full
   fix, flagged honestly below):
   - `rel_change`: each driver's late-period mean vs. early-period mean, computed as a
     **relative change on the raw (non-z-scored) series**, so it describes *how much a site's
     own forcing changed* rather than *where it sits in the training population* — a
     quantity that stays better-behaved even when the raw z-score is off the training scale.
   - `init_rate`: each initial-state variable divided by seed age, a proxy for "average
     accumulation rate to date," which is a more transferable quantity than the raw stock.
6. **An explicit extrapolation-risk diagnostic** (section 8) that reports what fraction of the
   real competition test features fall outside the range seen in training, per scenario. This
   won't fix extrapolation, but it tells you where to distrust the model most (usually: long
   seed ages, `ssp585`, late-period features) — worth a line in your report.

**Practical notes before you run this:**
- Building features for all ~54k sites x 15 ages (~812k rows x ~1,240 features) needs real
  RAM and time. If your Kaggle instance struggles, set `SITE_SUBSAMPLE_FRAC < 1.0` below, or
  drop `USE_ALL_LABELED_SITES` to `False` to fall back to the original train-only behavior.
- I was not able to execute this end-to-end myself — I don't have access to the competition's
  Kaggle dataset/environment. The shapes and logic follow the same conventions your v3
  notebook already used successfully (which did run through feature engineering), but please
  run it top to bottom yourself and tell me what breaks, if anything does; array-shape
  assumptions are the most likely source of a mismatch.


In [ ]:
# 0. Setup
!pip install -q lightgbm xgboost scikit-learn

import gc
import itertools
import random

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False
print('CatBoost available:', HAS_CATBOOST, '(optional — only used if present, nothing breaks without it)')

BASE = '/kaggle/input/datasets/zhihaow/carbonglobe'
# BASE = '/kaggle/input/carbonglobe'   # <- try this instead if the above 404s

SEED_AGES = [1, 10, 20, 30, 50, 70, 100, 150, 200, 250, 300, 350, 400, 450, 500]
AGE_FILES = {age: f"glob_Y{age:03d}.npy" for age in SEED_AGES}
TARGET_NAMES = ['height', 'agb', 'soil', 'lai', 'gpp', 'npp', 'rh']
SCALE = {'height': 10.90645, 'agb': 3.39567}

RANDOM_SEED = 0
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ---- knobs to trade compute for score; defaults chase leaderboard score ----
USE_ALL_LABELED_SITES = True   # False = reproduce v3's conservative train-only behavior
SITE_SUBSAMPLE_FRAC = 1.0      # lower this (e.g. 0.4) if you run out of RAM/time
NUM_FOLDS = 5                  # folds for both hyperparam CV and final bagging
HPO_SEARCH_SITES = 4000        # sites used for the (fast) hyperparameter search only


In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if 'glob_X_fea' in filename:
            print(os.path.join(dirname, filename))


## 1. Load data and build the full labeled-site pool

`train.csv` / `val.csv` / `test.csv` here are the CarbonGlobe **benchmark's own** split, used in
the NeurIPS paper to report a clean held-out number. For *this* Kaggle competition, the real,
private evaluation is entirely separate (the `ssp126`/`ssp585` future-climate files) — so there is
no leakage risk in also training on the benchmark's "val" and "test" sites. We just need our own
fresh group-based CV to pick hyperparameters and to get an honest score estimate; that CV is what
protects us here, not which of *their* three files a site happened to land in.

If you'd rather stay conservative (e.g. your assignment specifically wants you to respect the
provided split), set `USE_ALL_LABELED_SITES = False` above and this cell reproduces v3's behavior.


In [ ]:
X = np.load(f'{BASE}/data_global/glob_X_fea.npy', mmap_mode='r')
print('X shape:', X.shape)

train_sites = pd.read_csv(f'{BASE}/train.csv', header=None).iloc[:, 0].astype(int).values
val_sites   = pd.read_csv(f'{BASE}/val.csv',   header=None).iloc[:, 0].astype(int).values
test_sites  = pd.read_csv(f'{BASE}/test.csv',  header=None).iloc[:, 0].astype(int).values
print('benchmark split sizes -- train/val/test:', len(train_sites), len(val_sites), len(test_sites))

if USE_ALL_LABELED_SITES:
    all_sites = np.concatenate([train_sites, val_sites, test_sites])
else:
    all_sites = train_sites.copy()

all_sites = np.unique(all_sites)
if SITE_SUBSAMPLE_FRAC < 1.0:
    rng = np.random.RandomState(RANDOM_SEED)
    n_keep = int(len(all_sites) * SITE_SUBSAMPLE_FRAC)
    all_sites = rng.choice(all_sites, size=n_keep, replace=False)

print('final labeled-site pool used for training/CV:', len(all_sites),
      f'({len(all_sites) / 54152:.1%} of all sites)')

stats = np.load(f'{BASE}/data_stats/data_stats.npz')
x_mean, x_std = stats['x_mean'], stats['x_std']


## 2. Feature engineering v4

Kept from v3: population z-scored mean / linear trend / last-5-year mean / four decade means.

Added:
- `feat_std` — variability of each driver over the 40 years (climate variance, not just level).
- `rel_change` — computed on the **raw, non-z-scored** monthly forcing: `(late-period mean -
  early-period mean) / (|early-period mean| + eps)`. A relative, per-site quantity that should
  degrade more gracefully than a population z-score when future-scenario forcing (esp. `ssp585`
  CO2) sits outside anything seen in training.
- `init_rate` — each of the 7 initial-state variables divided by seed age: an "average rate of
  accumulation so far" feature, on top of the raw initial stock already in `init_state`.

Feature count grows from 960 to ~1,240 — kept the extra features targeted rather than exhaustive
(no full second-order interactions, no per-decade std) so the added overfitting surface stays
manageable; `feature_fraction` in the LightGBM/XGBoost params below is intentionally aggressive
(0.5-0.7) to compensate.


In [ ]:
EPS = 1e-6

def build_features_for_sites(site_idx_array, age):
    Y = np.load(f'{BASE}/data_global/{AGE_FILES[age]}', mmap_mode='r')

    Xf = np.array(X[site_idx_array])     # materialize the memmap slice once, raw units
    Yf = np.array(Y[site_idx_array])

    Xn = (Xf - x_mean) / (x_std + 1e-8)
    annual_mean = Xn.mean(axis=2)                         # (n, 40, 136) z-scored
    raw_annual_mean = Xf.mean(axis=2)                     # (n, 40, 136) raw units

    feat_mean  = annual_mean.mean(axis=1)
    feat_std   = annual_mean.std(axis=1)
    feat_last5 = annual_mean[:, -5:, :].mean(axis=1)

    years = np.arange(40)
    years_c = years - years.mean()
    denom = (years_c ** 2).sum()
    feat_trend = (annual_mean * years_c[None, :, None]).sum(axis=1) / denom

    decade_feats = []
    for d in range(4):
        decade_feats.append(annual_mean[:, d*10:(d+1)*10, :].mean(axis=1))   # (n, 136)
    decade_feats = np.concatenate(decade_feats, axis=1)                     # (n, 4*136)

    raw_early = raw_annual_mean[:, :5, :].mean(axis=1)
    raw_late  = raw_annual_mean[:, -5:, :].mean(axis=1)
    rel_change = (raw_late - raw_early) / (np.abs(raw_early) + EPS)
    rel_change = np.clip(rel_change, -10, 10)   # guard against divide-by-tiny-baseline blowups

    init_state = Yf[:, 0, -1, :]
    init_rate = init_state / (age + EPS)
    age_col = np.full((len(site_idx_array), 1), age, dtype=np.float32)

    feats = np.concatenate(
        [feat_mean, feat_std, feat_trend, feat_last5, decade_feats,
         rel_change, init_state, init_rate, age_col], axis=1
    ).astype(np.float32)

    target = Yf[:, 40, -1, :2]
    site_ids = np.array(site_idx_array)
    del Xf, Yf, Xn, raw_annual_mean
    return feats, target, site_ids


def build_feature_names():
    names = [f'x_mean_{i}' for i in range(136)]
    names += [f'x_std_{i}' for i in range(136)]
    names += [f'x_trend_{i}' for i in range(136)]
    names += [f'x_last5_{i}' for i in range(136)]
    for d in range(4):
        names += [f'x_decade{d}_{i}' for i in range(136)]
    names += [f'x_relchange_{i}' for i in range(136)]
    names += [f'init_{t}' for t in TARGET_NAMES]
    names += [f'initrate_{t}' for t in TARGET_NAMES]
    names += ['seed_age']
    return names

FEATURE_NAMES = build_feature_names()
print('n_features:', len(FEATURE_NAMES))


def assemble(site_idx_array, ages=SEED_AGES):
    """Preallocates the output array instead of concatenating a list of chunks, to keep peak
    memory down when site_idx_array is large (all ~54k sites x 15 ages)."""
    n_sites = len(site_idx_array)
    n_rows = n_sites * len(ages)
    n_feat = len(FEATURE_NAMES)

    X_out = np.empty((n_rows, n_feat), dtype=np.float32)
    y_out = np.empty((n_rows, 2), dtype=np.float32)
    sid_out = np.empty(n_rows, dtype=np.int64)

    for i, age in enumerate(ages):
        feats, target, sid = build_features_for_sites(site_idx_array, age)
        lo, hi = i * n_sites, (i + 1) * n_sites
        X_out[lo:hi] = feats
        y_out[lo:hi] = target
        sid_out[lo:hi] = sid
        del feats, target, sid
        gc.collect()
        print(f'  assembled age={age} ({i + 1}/{len(ages)})')

    return X_out, y_out, sid_out

print('Building features for', len(all_sites), 'sites x', len(SEED_AGES), 'ages'
      f' -> {len(all_sites) * len(SEED_AGES):,} rows x {len(FEATURE_NAMES)} features'
      ' (this is the slow, memory-heavy step)')
X_all, y_all, sid_all = assemble(all_sites)
print('X_all:', X_all.shape)


## 3. Hyperparameter search — grouped K-fold CV, same rigor for LightGBM and XGBoost

v3 only grid-searched LightGBM and left XGBoost on one fixed, comparatively weak set of
regularization params. Here both families get the same treatment: a moderate grid plus a
stronger, more heavily regularized grid searched specifically for `agb` (which overfit hardest
in earlier versions), all scored via grouped K-fold CV on a **subsample** of sites
(`HPO_SEARCH_SITES`) so the search itself stays fast — the final bagged models in section 4 are
what get trained on the full data.


In [ ]:
rng = np.random.RandomState(RANDOM_SEED)
if len(all_sites) > HPO_SEARCH_SITES:
    hpo_site_pool = rng.choice(all_sites, size=HPO_SEARCH_SITES, replace=False)
else:
    hpo_site_pool = all_sites
hpo_mask = np.isin(sid_all, hpo_site_pool)
X_hpo, y_hpo, sid_hpo = X_all[hpo_mask], y_all[hpo_mask], sid_all[hpo_mask]
print('HPO subsample:', X_hpo.shape)

LGB_GRID = [
    dict(num_leaves=nl, min_data_in_leaf=md_, lambda_l1=l1, lambda_l2=l2,
         feature_fraction=ff, max_depth=depth)
    for nl, md_, l1, l2, ff, depth in itertools.product(
        [15, 31], [30, 80], [0.0, 0.5], [0.5, 1.0], [0.6, 0.8], [-1, 6])
]
LGB_GRID_STRONG = [
    dict(num_leaves=nl, min_data_in_leaf=md_, lambda_l1=l1, lambda_l2=l2,
         feature_fraction=ff, max_depth=depth)
    for nl, md_, l1, l2, ff, depth in itertools.product(
        [7, 15], [150, 300], [1.0, 2.0], [2.0, 5.0], [0.5, 0.6], [4, 6])
]

XGB_GRID = [
    dict(max_depth=depth, min_child_weight=mcw, reg_alpha=a, reg_lambda=l, subsample=ss, colsample_bytree=cs)
    for depth, mcw, a, l, ss, cs in itertools.product(
        [4, 6], [5, 20], [0.0, 0.5], [1.0, 2.0], [0.7, 0.8], [0.6, 0.8])
]
XGB_GRID_STRONG = [
    dict(max_depth=depth, min_child_weight=mcw, reg_alpha=a, reg_lambda=l, subsample=ss, colsample_bytree=cs)
    for depth, mcw, a, l, ss, cs in itertools.product(
        [2, 3], [40, 80], [1.0, 2.0], [3.0, 5.0], [0.6, 0.7], [0.5, 0.6])
]

def lgb_cv_score(X_tr, y_tr, groups, params, n_splits=4, num_boost_round=1200):
    gkf = GroupKFold(n_splits=n_splits)
    fold_rmses = []
    for tr_idx, va_idx in gkf.split(X_tr, groups=groups):
        train_set = lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx])
        val_set = lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx], reference=train_set)
        full_params = dict(objective='regression', metric='rmse', learning_rate=0.05,
                            bagging_fraction=0.8, bagging_freq=1, verbose=-1, **params)
        model = lgb.train(full_params, train_set, num_boost_round=num_boost_round,
                           valid_sets=[val_set],
                           callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        pred = model.predict(X_tr[va_idx])
        fold_rmses.append(mean_squared_error(y_tr[va_idx], pred) ** 0.5)
    return np.mean(fold_rmses)

def xgb_cv_score(X_tr, y_tr, groups, params, n_splits=4, num_boost_round=1200):
    gkf = GroupKFold(n_splits=n_splits)
    fold_rmses = []
    for tr_idx, va_idx in gkf.split(X_tr, groups=groups):
        dtrain = xgb.DMatrix(X_tr[tr_idx], label=y_tr[tr_idx])
        dval = xgb.DMatrix(X_tr[va_idx], label=y_tr[va_idx])
        full_params = dict(objective='reg:squarederror', eta=0.05, eval_metric='rmse', **params)
        model = xgb.train(full_params, dtrain, num_boost_round=num_boost_round,
                           evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
        pred = model.predict(dval, iteration_range=(0, model.best_iteration + 1))
        fold_rmses.append(mean_squared_error(y_tr[va_idx], pred) ** 0.5)
    return np.mean(fold_rmses)

def search_best(cv_fn, grid_moderate, grid_strong, target_idx, target_name, n_moderate, n_strong):
    grid = random.sample(grid_moderate, min(n_moderate, len(grid_moderate)))
    if target_name == 'agb':
        grid += random.sample(grid_strong, min(n_strong, len(grid_strong)))
    print(f'  searching {len(grid)} combos for {target_name}')
    scores = []
    for params in grid:
        s = cv_fn(X_hpo, y_hpo[:, target_idx], sid_hpo, params)
        scores.append((s, params))
    scores.sort(key=lambda t: t[0])
    print(f'  BEST {target_name}: CV RMSE {scores[0][0]:.4f} | {scores[0][1]}')
    return scores[0][1]

best_lgb_params, best_xgb_params = {}, {}
for target_idx, target_name in enumerate(['height', 'agb']):
    print(f'=== LightGBM: tuning {target_name} ===')
    best_lgb_params[target_name] = search_best(lgb_cv_score, LGB_GRID, LGB_GRID_STRONG,
                                                target_idx, target_name, n_moderate=8, n_strong=8)
    print(f'=== XGBoost: tuning {target_name} ===')
    best_xgb_params[target_name] = search_best(xgb_cv_score, XGB_GRID, XGB_GRID_STRONG,
                                                target_idx, target_name, n_moderate=8, n_strong=8)

del X_hpo, y_hpo, sid_hpo
gc.collect()


## 4. Final training: K-fold bagging on the full labeled-site pool

For each target (`height`, `agb`) and each model family (LightGBM, XGBoost), train `NUM_FOLDS`
models on grouped folds of **all** labeled sites. Each fold's held-out sites are used for early
stopping and to produce an **out-of-fold (OOF) prediction** — a prediction from a model that
never saw that site — so the OOF array covers the whole labeled pool with zero leakage. At test
time (section 9) all `NUM_FOLDS` models per family are averaged together (bagging).


In [ ]:
gkf = GroupKFold(n_splits=NUM_FOLDS)
fold_indices = list(gkf.split(X_all, groups=sid_all))

lgb_models = {'height': [], 'agb': []}
xgb_models = {'height': [], 'agb': []}
oof_lgb = {'height': np.zeros(len(y_all)), 'agb': np.zeros(len(y_all))}
oof_xgb = {'height': np.zeros(len(y_all)), 'agb': np.zeros(len(y_all))}

for target_idx, target_name in enumerate(['height', 'agb']):
    y_t = y_all[:, target_idx]
    for fold_i, (tr_idx, va_idx) in enumerate(fold_indices):
        # LightGBM
        train_set = lgb.Dataset(X_all[tr_idx], label=y_t[tr_idx], feature_name=FEATURE_NAMES)
        val_set = lgb.Dataset(X_all[va_idx], label=y_t[va_idx], reference=train_set)
        lgb_params = dict(objective='regression', metric='rmse', learning_rate=0.03,
                           bagging_fraction=0.8, bagging_freq=1, verbose=-1,
                           **best_lgb_params[target_name])
        model_lgb = lgb.train(lgb_params, train_set, num_boost_round=4000,
                               valid_sets=[val_set], valid_names=['val'],
                               callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)])
        oof_lgb[target_name][va_idx] = model_lgb.predict(X_all[va_idx])
        lgb_models[target_name].append(model_lgb)

        # XGBoost
        dtrain = xgb.DMatrix(X_all[tr_idx], label=y_t[tr_idx])
        dval = xgb.DMatrix(X_all[va_idx], label=y_t[va_idx])
        xgb_params = dict(objective='reg:squarederror', eta=0.03, eval_metric='rmse',
                           **best_xgb_params[target_name])
        model_xgb = xgb.train(xgb_params, dtrain, num_boost_round=4000,
                               evals=[(dval, 'val')], early_stopping_rounds=150, verbose_eval=False)
        oof_xgb[target_name][va_idx] = model_xgb.predict(
            dval, iteration_range=(0, model_xgb.best_iteration + 1))
        xgb_models[target_name].append(model_xgb)

        print(f'{target_name} fold {fold_i + 1}/{NUM_FOLDS} done '
              f'(lgb best_iter={model_lgb.best_iteration}, xgb best_iter={model_xgb.best_iteration})')

    gc.collect()


## 5. Overfitting check + blend-weight search — all on out-of-fold predictions

Both `check_overfit` and the blend search now run on OOF predictions, i.e. no model ever
predicts a site it trained on. This also now checks **both** model families, not just LightGBM
like v3 did.


In [ ]:
def check_overfit_oof(fold_models, X_full, y_full, fold_indices, oof_pred, name):
    tr_rmses = []
    for (tr_idx, _), model in zip(fold_indices, fold_models):
        pred_tr = model.predict(X_full[tr_idx]) if hasattr(model, 'predict') and not isinstance(model, xgb.Booster) \
            else model.predict(xgb.DMatrix(X_full[tr_idx]))
        tr_rmses.append(mean_squared_error(y_full[tr_idx], pred_tr) ** 0.5)
    tr_rmse = np.mean(tr_rmses)
    oof_rmse = mean_squared_error(y_full, oof_pred) ** 0.5
    ratio = oof_rmse / (tr_rmse + 1e-9)
    print(f'{name}: mean train RMSE {tr_rmse:.4f} | OOF RMSE {oof_rmse:.4f} | OOF/train ratio {ratio:.2f}'
          + ('  <-- still overfitting, consider more regularization' if ratio > 3 else '  OK'))

for target_name in ['height', 'agb']:
    check_overfit_oof(lgb_models[target_name], X_all, y_all[:, ['height', 'agb'].index(target_name)],
                       fold_indices, oof_lgb[target_name], f'LGB {target_name}')
    check_overfit_oof(xgb_models[target_name], X_all, y_all[:, ['height', 'agb'].index(target_name)],
                       fold_indices, oof_xgb[target_name], f'XGB {target_name}')

def blend_and_score_oof(target_idx, target_name):
    lgb_pred = oof_lgb[target_name]
    xgb_pred = oof_xgb[target_name]
    y_t = y_all[:, target_idx]

    best_w, best_mse = 0.5, np.inf
    for w in np.linspace(0, 1, 41):
        blend = w * lgb_pred + (1 - w) * xgb_pred
        scaled_mse = mean_squared_error(y_t / SCALE[target_name], blend / SCALE[target_name])
        if scaled_mse < best_mse:
            best_mse, best_w = scaled_mse, w

    blend = best_w * lgb_pred + (1 - best_w) * xgb_pred
    rmse = mean_squared_error(y_t, blend) ** 0.5
    mae = mean_absolute_error(y_t, blend)
    r2 = r2_score(y_t, blend)
    print(f'--- {target_name} OOF blend (lgb weight={best_w:.2f}) ---')
    print(f'RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}  scaled MSE: {best_mse:.6f}')
    return blend, best_w, best_mse

blend_height, w_height, mse_h = blend_and_score_oof(0, 'height')
blend_agb, w_agb, mse_a = blend_and_score_oof(1, 'agb')

final_score = 0.5 * (mse_h + mse_a)
print(f'\nEstimated (honest, OOF) competition-style score: {final_score:.6f}'
      f'  (relative RMSE {final_score ** 0.5:.6f})')


## 6. Feature importance (averaged across the LightGBM fold models)

In [ ]:
def plot_importance_avg(models, title, top_n=20):
    imp = np.mean([m.feature_importance(importance_type='gain') for m in models], axis=0)
    imp = pd.Series(imp, index=FEATURE_NAMES).sort_values(ascending=False).head(top_n)
    plt.figure(figsize=(8, 6))
    imp[::-1].plot(kind='barh')
    plt.title(f'Top {top_n} features — {title} (mean gain across {len(models)} folds)')
    plt.xlabel('Gain')
    plt.tight_layout()
    plt.show()

plot_importance_avg(lgb_models['height'], 'height')
plot_importance_avg(lgb_models['agb'], 'agb')


## 7. Error vs. seed age (on OOF predictions, so it reflects real generalization)

In [ ]:
age_col_idx = FEATURE_NAMES.index('seed_age')
all_ages_col = X_all[:, age_col_idx]
results = []
for age in SEED_AGES:
    mask = all_ages_col == age
    if mask.sum() == 0:
        continue
    mse_h_age = mean_squared_error(y_all[mask, 0] / SCALE['height'], blend_height[mask] / SCALE['height'])
    mse_a_age = mean_squared_error(y_all[mask, 1] / SCALE['agb'], blend_agb[mask] / SCALE['agb'])
    results.append((age, mask.sum(), 0.5 * (mse_h_age + mse_a_age)))

res_df = pd.DataFrame(results, columns=['seed_age', 'n_samples', 'scaled_mse'])
print(res_df)

plt.figure(figsize=(7, 4))
plt.plot(res_df['seed_age'], res_df['scaled_mse'], marker='o')
plt.xlabel('Seed forest age (years)')
plt.ylabel('Scaled MSE (OOF blend)')
plt.title('OOF error vs. seed forest age')
plt.tight_layout()
plt.show()


## 8. Extrapolation-risk diagnostic

Tree ensembles cannot extrapolate past the feature ranges seen in training. This cell reports,
per test scenario, what fraction of test rows fall outside the training feature range — a cheap,
honest way to say "trust these predictions less" for the flagged rows (usually: high seed ages
and `ssp585`, the high-emissions scenario). This doesn't fix the underlying limitation, just
surfaces it — worth a sentence in your report either way.


In [ ]:
train_min = X_all.min(axis=0)
train_max = X_all.max(axis=0)

def extrapolation_report(test_feats, scenario_name):
    below = test_feats < train_min[None, :]
    above = test_feats > train_max[None, :]
    out_of_range = below | above
    frac_rows_flagged = (out_of_range.any(axis=1)).mean()
    frac_cells_flagged = out_of_range.mean()
    worst_features = pd.Series(out_of_range.mean(axis=0), index=FEATURE_NAMES) \
        .sort_values(ascending=False).head(10)
    print(f'--- {scenario_name} ---')
    print(f'{frac_rows_flagged:.1%} of rows have at least one out-of-training-range feature')
    print(f'{frac_cells_flagged:.2%} of all (row, feature) cells are out of range')
    print('most-affected features:')
    print(worst_features)
    print()


## 9. Generate the official competition submission (bagged across all folds and both model families)

Matches the exact spec: one row per `(scenario, site, seed_age)`, `id = "{scenario}_{site}_{age}"`,
predicted year-40 `height` divided by `10.90645` and `agb` divided by `3.39567`. Uses the
**competition's** test files (`test_ssp126.npz`/`test_ssp585.npz`, `sites_ssp.csv`,
`sample_submission.csv`) — attach the competition dataset to this notebook too before running
this cell. Every one of the `NUM_FOLDS` LightGBM models and `NUM_FOLDS` XGBoost models is used
and averaged — this is the bagging payoff from section 4.


In [ ]:
TEST_BASE = '/kaggle/input/ieee-bigdata-cup-2026-ai-emulation-challenge'  # adjust to your actual mount path

sites_ssp = pd.read_csv(f'{TEST_BASE}/public/sites_ssp.csv')
print(sites_ssp.head())          # confirm the actual site-id column name before trusting iloc[:, 0]
site_ids = sites_ssp.iloc[:, 0].astype(int).values   # force int to avoid a "0.0" vs "0" id mismatch

def featurize_test(npz_path):
    d = np.load(npz_path)
    test_x, test_y0 = d['test_x'], d['test_y0']
    n_sites = test_x.shape[0]

    Xn = (test_x - x_mean) / (x_std + 1e-8)
    raw_annual_mean = test_x.mean(axis=2)
    annual_mean = Xn.mean(axis=2)

    feat_mean = annual_mean.mean(axis=1)
    feat_std = annual_mean.std(axis=1)
    feat_last5 = annual_mean[:, -5:, :].mean(axis=1)
    years = np.arange(40); years_c = years - years.mean(); denom = (years_c ** 2).sum()
    feat_trend = (annual_mean * years_c[None, :, None]).sum(axis=1) / denom
    decade_feats = np.concatenate(
        [annual_mean[:, d_*10:(d_+1)*10, :].mean(axis=1) for d_ in range(4)], axis=1)
    raw_early = raw_annual_mean[:, :5, :].mean(axis=1)
    raw_late = raw_annual_mean[:, -5:, :].mean(axis=1)
    rel_change = np.clip((raw_late - raw_early) / (np.abs(raw_early) + EPS), -10, 10)

    rows = []
    all_feats_for_report = []
    for age_i, age in enumerate(SEED_AGES):
        init_state = test_y0[:, age_i, :]
        init_rate = init_state / (age + EPS)
        age_col = np.full((n_sites, 1), age, dtype=np.float32)
        feats = np.concatenate(
            [feat_mean, feat_std, feat_trend, feat_last5, decade_feats,
             rel_change, init_state, init_rate, age_col], axis=1
        ).astype(np.float32)
        all_feats_for_report.append(feats)

        lgb_h = np.mean([m.predict(feats) for m in lgb_models['height']], axis=0)
        xgb_h = np.mean([m.predict(xgb.DMatrix(feats)) for m in xgb_models['height']], axis=0)
        lgb_a = np.mean([m.predict(feats) for m in lgb_models['agb']], axis=0)
        xgb_a = np.mean([m.predict(xgb.DMatrix(feats)) for m in xgb_models['agb']], axis=0)

        pred_h = (w_height * lgb_h + (1 - w_height) * xgb_h) / SCALE['height']
        pred_a = (w_agb * lgb_a + (1 - w_agb) * xgb_a) / SCALE['agb']

        rows.append(pd.DataFrame({'site': site_ids, 'age': age, 'height': pred_h, 'agb': pred_a}))

    return pd.concat(rows, ignore_index=True), np.concatenate(all_feats_for_report, axis=0)

df_126, feats_126 = featurize_test(f'{TEST_BASE}/public/test_ssp126.npz')
df_126['id'] = 'ssp126_' + df_126['site'].astype(str) + '_' + df_126['age'].astype(str)
extrapolation_report(feats_126, 'ssp126 (low emissions)')

df_585, feats_585 = featurize_test(f'{TEST_BASE}/public/test_ssp585.npz')
df_585['id'] = 'ssp585_' + df_585['site'].astype(str) + '_' + df_585['age'].astype(str)
extrapolation_report(feats_585, 'ssp585 (high emissions)')

submission = pd.concat([df_126, df_585], ignore_index=True)[['id', 'height', 'agb']]

# align exactly to sample_submission.csv's row order -- don't trust your own concat order
sample_sub = pd.read_csv(f'{TEST_BASE}/public/sample_submission.csv')
submission = sample_sub[['id']].merge(submission, on='id', how='left')

assert submission.shape[0] == 185130, f'expected 185130 rows, got {submission.shape[0]}'
assert submission[['height', 'agb']].isna().sum().sum() == 0, 'missing predictions for some ids -- id string mismatch!'

submission.to_csv('submission.csv', index=False)
submission.head()
